# IIP314W Optimización Aplicada a Negocios - 2026-T1
## Ayudantía 4: Modelamiento

### Ejercicio 1: Modelamiento

Una exportadora de frutas frescas necesita diseñar su plan de distribución para la próxima temporada de $T$ semanas. La empresa cuenta con $I$ plantas de empaque y $J$ centros de distribución (CD) en el extranjero. Debido a la naturaleza de la fruta, una fracción $\gamma$ de la mercadería almacenada se echa a perder cada semana. Cada planta $i$ tiene una capacidad máxima de empaque por semana ($cap_p$ de la planta $i$). Existen $M$ modos de transporte (ej: Rápido, Estándar). Cada modo $m$ tiene un costo unitario $c_{ijm}$ y un tiempo de tránsito despreciable dentro de la misma semana. Cada CD $j$ tiene una demanda proyectada $d_{jt}$ para cada semana $t$, la cual debe satisfacerse obligatoriamente. Los CDs pueden almacenar fruta con un costo unitario por semana $h_j$. Sin embargo, la fruta es perecible: el inventario que queda al final de la semana $t-1$ se reduce en un factor $\gamma$ antes de estar disponible para la semana $t$. Además, cada CD $j$ tiene un límite físico para el inventario final de cada semana ($cap_w$ del centro $j$).

Formule un modelo de programación lineal que minimice los costos totales de la operación. Defina claramente conjuntos, parámetros, variables de decisión, función objetivo y restricciones.

#### Solución propuesta:

**Conjuntos:**
- $i \in I$: Plantas de empaque.
- $j \in J$: Centros de distribución (CD).
- $t \in \{1, \dots, T\}$: Semanas de la temporada.
- $m \in M$: Modos de transporte.

**Parámetros:**
- $d_{jt}$: Demanda en CD $j$ durante la semana $t$ (unidades).
- $c_{ijm}$: Costo unitario de transportar desde $i$ a $j$ mediante el modo $m$ (\$/unidad).
- $h_j$: Costo unitario de mantener inventario en el CD $j$ (\$/unidad/semana).
- $cap_{p,i}$: Capacidad de producción/empaque semanal de la planta $i$ (unidades).
- $cap_{w,j}$: Capacidad de almacenamiento del CD $j$ (unidades).
- $\gamma$: Tasa de pérdida de inventario por perecibilidad ($0 \le \gamma \le 1$).

**Variables de Decisión:**
- $x_{ijmt} \ge 0$: Cantidad de fruta enviada desde $i$ a $j$ vía modo $m$ en la semana $t$.
- $inv_{jt} \ge 0$: Inventario de fruta en el CD $j$ al final de la semana $t$.

**Función Objetivo:**
Minimizar los costos totales de transporte y almacenamiento:
$$
\min Z = \sum_{i \in I} \sum_{j \in J} \sum_{m \in M} \sum_{t=1}^T c_{ijm} x_{ijmt} + \sum_{j \in J} \sum_{t=1}^T h_j inv_{jt}
$$

**Restricciones:**
1. **Balance de Inventario en CDs:**
$$
inv_{j, t-1} \cdot (1 - \gamma) + \sum_{i \in I} \sum_{m \in M} x_{ijmt} = d_{jt} + inv_{jt} \quad \forall j, t
$$
(Asumiendo $inv_{j,0} = 0$)

2. **Capacidad de Empaque (Plantas):**
$$
\sum_{j \in J} \sum_{m \in M} x_{ijmt} \le cap_{p,i} \quad \forall i, t
$$

3. **Capacidad de Bodega (CDs):**
$$
inv_{jt} \le cap_{w,j} \quad \forall j, t
$$

4. **Naturaleza de variables:**
$$
x_{ijmt}, inv_{jt} \ge 0
$$

---
### Ejercicio 2: Modelamiento

**Escenario:**
La ciudad de "Opti-Ville" necesita planificar su generación eléctrica diaria para satisfacer una demanda de **800 MW**. Dispone de 4 tipos de tecnologías con las características detalladas en la tabla:

| Tecnología | Costo Variable ($/MWh) | Emisiones (kg CO2/MWh) | Capacidad Máx (MW) |
|:---|:---:|:---:|:---:|
| Carbón | 40 | 900 | 500 |
| Gas Natural | 60 | 450 | 300 |
| Solar | 20 | 0 | 200 |
| Eólico | 25 | 0 | 150 |

**Restricciones Adicionales:**
1. **Meta de Renovables:** Por ley ambiental, al menos el **25%** de la energía generada debe provenir de fuentes renovables (Solar + Eólico).
2. **Tope de Emisiones:** La ciudad no puede emitir más de **400 toneladas de CO2** al día.

**Se pide:**

**a) Modelamiento:**

**Variables:**
- $x_1$: Energía generada mediante Carbón (MW).
- $x_2$: Energía generada mediante Gas Natural (MW).
- $x_3$: Energía generada mediante Solar (MW).
- $x_4$: Energía generada mediante Eólico (MW).

**Función Objetivo (Costo Operativo Total):**
$$
\min Z = 40x_1 + 60x_2 + 20x_3 + 25x_4
$$

**Restricciones:**
1. **Satisfacción de Demanda:**
$$
x_1 + x_2 + x_3 + x_4 = 800
$$
2. **Meta de Renovables:**
$$
x_3 + x_4 \ge 0.25 \cdot 800
$$
3. **Tope de Emisiones:**
$$
900x_1 + 450x_2 \le 400.000
$$
4. **Capacidades Máximas:**
$$
x_1 \le 500, \quad x_2 \le 300, \quad x_3 \le 200, \quad x_4 \le 150
$$
5. **No negatividad:**
$$
x_1, x_2, x_3, x_4 \ge 0
$$

**b)** Resuelva el problema utilizando `scipy.optimize.minimize` y reporte la cantidad de MW a producir con cada tecnología.

In [4]:
import numpy as np
from scipy.optimize import minimize

# b) Desarrollo en código

fo = lambda x: 40*x[0] + 60*x[1] + 20*x[2] + 25*x[3]
demanda = lambda x: x[0] + x[1] + x[2] + x[3] - 800
renovable = lambda x: (x[2] + x[3]) - 0.25 * 800
carbono = lambda x: 400000 - (900*x[0] + 450*x[1])

# Límites de capacidad (Bounds)
b_carbon = (0, 500)
b_gas = (0, 300)
b_solar = (0, 200)
b_eolica = (0, 150)
bounds = [b_carbon, b_gas, b_solar, b_eolica]

con1 = {'type': 'ineq', 'fun': demanda}
con2 = {'type': 'ineq', 'fun': renovable}
con3 = {'type': 'ineq', 'fun': carbono}
cons = [con1, con2, con3]

# Resolución
x0 = [200, 200, 200, 200]
sol = minimize(fo, x0, method='SLSQP', bounds=bounds, constraints=cons)

print("----- RESULTADOS MIX ENERGÉTICO -----")
technologies = ["Carbón", "Gas Natural", "Solar", "Eólico"]
for i in range(4):
    print(f"{technologies[i]}: {sol.x[i]:.2f} MW")

print(f"\nCosto Total: ${sol.fun:.2f}")
print(f"Emisiones Totales: {(900*sol.x[0] + 450*sol.x[1])/1000:.2f} toneladas de CO2")

----- RESULTADOS MIX ENERGÉTICO -----
Carbón: 438.89 MW
Gas Natural: 11.11 MW
Solar: 200.00 MW
Eólico: 150.00 MW

Costo Total: $25972.22
Emisiones Totales: 400.00 toneladas de CO2


---
### Ejercicio 3: Modelamiento con Gurobi

**Escenario:**
Una empresa de tecnología produce 3 modelos de sensores inteligentes ($S_1, S_2, S_3$). Cada modelo tiene un ingreso por unidad, requiere tiempo de ensamblaje en una línea de producción compartida y tiene un **costo fijo de activación** de la maquinaria específica para ese modelo, además de un **costo variable** de producción por unidad.

| Sensor | Ingreso ($/un) | Tiempo (hr/un) | Costo Fijo ($) | Costo Var ($/un) | Demanda Máx (un) |
|:---:|:---:|:---:|:---:|:---:|:---:|
| S1 | 150 | 2 | 5000 | 100 | 200 |
| S2 | 230 | 3 | 7000 | 150 | 150 |
| S3 | 350 | 5 | 10000 | 230 | 100 |

**Restricciones:**
1. **Línea de producción:** Tiempo total disponible de **600 horas**.
2. **Lote Mínimo:** Si se activa la producción de un sensor, se debe producir al menos un **mínimo de 30 unidades**.
3. **Presupuesto Fijo:** El presupuesto total para costos fijos de activación es de **$20,000**.

**Se pide:**

**a) Modelamiento:**
**Variables:**
- $x_i$: Unidades producidas del sensor $i \in \{1, 2, 3\}$ (Continua).
- $y_i$: Variable binaria; 1 si se activa la producción del sensor $i$, 0 en caso contrario.

**Función Objetivo (Maximizar Utilidad Neta):**
$$\max Z = \sum_{i=1}^3 ((Ingreso_i - CostoVar_i) \cdot x_i - CostoFijo_i \cdot y_i)$$

**Restricciones:**
1. **Capacidad de Tiempo:** $\sum_{i=1}^3 Tiempo_i \cdot x_i \le 600$
2. **Activación y Demanda (Big-M):** $x_i \le DemandaMax_i \cdot y_i \quad \forall i$
3. **Lote Mínimo:** $x_i \ge 30 \cdot y_i \quad \forall i$
4. **Presupuesto de Activación:** $\sum_{i=1}^3 CostoFijo_i \cdot y_i \le 20.000$
5. **No negatividad y Naturaleza:** $x_i \ge 0, y_i \in \{0, 1\}$

**b)** Implemente y resuelva el modelo utilizando la librería `gurobipy`.

In [2]:
import gurobipy as gp
from gurobipy import GRB

# b) Desarrollo con Gurobi

# Datos
sensores = ['S1', 'S2', 'S3']
ingreso = {'S1': 150, 'S2': 230, 'S3': 350}
tiempo = {'S1': 2, 'S2': 3, 'S3': 5}
costo_fijo = {'S1': 5000, 'S2': 7000, 'S3': 10000}
costo_var = {'S1': 100, 'S2': 150, 'S3': 230}
demanda_max = {'S1': 200, 'S2': 150, 'S3': 100}

# Modelo
m = gp.Model("Produccion_Sensores")

# Variables
x = m.addVars(sensores, lb=0, vtype=GRB.CONTINUOUS, name="x")
y = m.addVars(sensores, vtype=GRB.BINARY, name="y")

# FO
m.setObjective(gp.quicksum((ingreso[i]-costo_var[i])*x[i] - costo_fijo[i]*y[i] for i in sensores), GRB.MAXIMIZE)

# Restricciones
m.addConstr(gp.quicksum(tiempo[i]*x[i] for i in sensores) <= 600, "Capacidad")
m.addConstrs((x[i] <= demanda_max[i]*y[i] for i in sensores), "Activacion_Demanda")
m.addConstrs((x[i] >= 30*y[i] for i in sensores), "Lote_Minimo")
m.addConstr(gp.quicksum(costo_fijo[i]*y[i] for i in sensores) <= 20000, "Presupuesto")

# Optimizar
m.optimize()

if m.status == GRB.OPTIMAL:
    print("\n----- RESULTADOS GUROBI -----")
    for i in sensores:
        if y[i].X:
            print(f"{i}: {x[i].X:.2f} unidades")
    print(f"\nUtilidad Total Neta: ${m.objVal:.2f}")

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 10.0 (19045.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 8 rows, 6 columns and 18 nonzeros
Model fingerprint: 0x292977db
Variable types: 3 continuous, 3 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+04]
  Objective range  [5e+01, 1e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [6e+02, 2e+04]
Found heuristic solution: objective -0.0000000
Presolve removed 3 rows and 0 columns
Presolve time: 0.00s
Presolved: 5 rows, 6 columns, 15 nonzeros
Variable types: 3 continuous, 3 integer (3 binary)
Found heuristic solution: objective 5000.0000000

Root relaxation: objective 7.222222e+03, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf 